# KantBench: GRPO Training on 90+ Game Theory Environments

Train a language model to play strategic games optimally using **Group Relative Policy Optimization (GRPO)** via HF TRL.

**How it works:**
- 90+ game theory environments (Prisoner's Dilemma, Cournot, Auctions, Signaling, ...)
- 17 opponent strategies (tit-for-tat, grudger, adaptive, ...)
- Each LLM completion is a **move** — the reward function plays a **full multi-round episode** using that move as the agent's strategy
- Composite reward: payoff + cooperation rate + Pareto efficiency + fairness

**Requirements:** Colab GPU runtime (T4 for 1.5B, A100 for 3B+)

In [ ]:
!pip install -q torch transformers trl datasets accelerate peft openenv-core>=0.2.1 wandb bitsandbytes nest_asyncio

In [ ]:
# Clone the repo to get the full game registry
!git clone --depth 1 https://github.com/wisent-ai/OpenEnv.git /content/OpenEnv
import sys
sys.path.insert(0, "/content/OpenEnv")

In [ ]:
import wandb
wandb.login()

## Config

In [ ]:
# --- Adjust these for your GPU ---
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"  # 1.5B fits on T4; use 3B on A100
NUM_EPISODES = 500
NUM_GENERATIONS = 4
BATCH_SIZE = 1
GRAD_ACCUM = 8
MAX_STEPS = 200
LR = 5e-6

## Load Environment

In [ ]:
import random
from common.games import GAMES
from common.strategies import STRATEGIES as STRATEGY_REGISTRY
from env.environment import KantEnvironment
from env.models import GameAction, GameObservation
from train.agent import PromptBuilder, parse_action
from train.rewards import episode_reward
from train.trajectory import _compute_cooperation_rate

print(f"Loaded {len(GAMES)} games, {len(STRATEGY_REGISTRY)} strategies")
print(f"Sample games: {list(GAMES.keys())[:10]}")

## Build Dataset with Real Environment States

Uses `PromptBuilder` for structured prompts and simulates partial game histories
so the model trains on diverse game states (not just round 1).

In [ ]:
from datasets import Dataset

SYSTEM_PROMPT = (
    "You are playing a game-theory game. Analyse the situation and choose "
    "the best action. Respond with ONLY the action name, nothing else."
)

def build_dataset(n_samples):
    env = KantEnvironment()
    game_keys = list(GAMES.keys())
    strat_names = list(STRATEGY_REGISTRY.keys())
    prompt_builder = PromptBuilder()
    samples = []

    for _ in range(n_samples):
        game_key = random.choice(game_keys)
        strategy = random.choice(strat_names)

        obs = env.reset(game=game_key, strategy=strategy)

        # Play 0..N-1 random rounds for diverse game states
        rounds_to_play = random.randint(0, max(obs.total_rounds - 1, 0))
        for _ in range(rounds_to_play):
            random_action = GameAction(action=random.choice(obs.available_actions))
            obs = env.step(random_action)
            if obs.done:
                break

        if obs.done:
            obs = env.reset(game=game_key, strategy=strategy)

        prompt = prompt_builder.build(obs)
        samples.append({
            "prompt": prompt,
            "game_key": game_key,
            "strategy": strategy,
            "available_moves": list(obs.available_actions),
        })

    return Dataset.from_list(samples)


dataset = build_dataset(NUM_EPISODES)
print(f"Dataset: {len(dataset)} prompts")
print(f"\nSample prompt:\n{dataset[0]['prompt'][:500]}")

## Reward Function: Full Episode Rollout

For each LLM completion:
1. Parse the move
2. Play a **full multi-round episode** using that move as the agent's strategy
3. Compute composite reward: payoff + cooperation + Pareto efficiency + fairness

In [ ]:
from typing import Any

reward_env = KantEnvironment()

def kantbench_reward(completions: list[str], prompts: list[str], **kwargs: Any) -> list[float]:
    rewards = []
    game_keys = kwargs.get("game_key", ["prisoners_dilemma"] * len(completions))
    strategies = kwargs.get("strategy", ["tit_for_tat"] * len(completions))
    available_moves_batch = kwargs.get("available_moves", [["cooperate", "defect"]] * len(completions))

    for completion, game_key, strategy, moves in zip(
        completions, game_keys, strategies, available_moves_batch
    ):
        action_str = parse_action(completion.strip(), moves)

        try:
            # Full episode rollout
            obs = reward_env.reset(game=game_key, strategy=strategy)
            while not obs.done:
                obs = reward_env.step(GameAction(action=action_str))

            coop_rate = _compute_cooperation_rate(obs)
            reward = episode_reward(
                player_score=obs.player_score,
                opponent_score=obs.opponent_score,
                cooperation_rate=coop_rate,
                total_rounds=obs.current_round,
            )
            rewards.append(reward)
        except Exception as e:
            rewards.append(-1.0)

    return rewards


# Sanity check — cooperate vs defect in PD
for move in ["cooperate", "defect"]:
    r = kantbench_reward(
        [move], ["..."],
        game_key=["prisoners_dilemma"],
        strategy=["tit_for_tat"],
        available_moves=[["cooperate", "defect"]],
    )
    print(f"PD vs tit_for_tat | {move:10s} -> composite reward = {r[0]:.3f}")

## Train with GRPO

In [ ]:
import torch
from transformers import AutoTokenizer
from trl import GRPOConfig, GRPOTrainer

tokenizer = AutoTokenizer.from_pretrained(MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def format_prompt(example):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": example["prompt"]},
    ]
    return {"prompt": tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )}

train_dataset = dataset.map(format_prompt)

config = GRPOConfig(
    output_dir="/content/kantbench-grpo",
    num_generations=NUM_GENERATIONS,
    max_completion_length=16,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    max_steps=MAX_STEPS,
    logging_steps=5,
    save_steps=50,
    bf16=torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8,
    fp16=torch.cuda.is_available() and torch.cuda.get_device_capability()[0] < 8,
    report_to="wandb",
)

trainer = GRPOTrainer(
    model=MODEL,
    reward_funcs=kantbench_reward,
    args=config,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)

print(f"Training {MODEL} on {len(GAMES)} games with GRPO")
print(f"Reward: full-episode composite (payoff + cooperation + Pareto + fairness)")
trainer.train()

In [ ]:
trainer.save_model("/content/kantbench-grpo")
print("Model saved to /content/kantbench-grpo")

## Evaluate: Before vs After

In [ ]:
from transformers import pipeline

test_games = ["prisoners_dilemma", "stag_hunt", "hawk_dove", "cournot", "battle_of_the_sexes"]
prompt_builder = PromptBuilder()
eval_env = KantEnvironment()

pipe = pipeline("text-generation", model="/content/kantbench-grpo", tokenizer=tokenizer,
                max_new_tokens=8, do_sample=False)

print("=" * 70)
print(f"{'Game':<30s} {'Move':<15s} {'Episode Reward':>15s}")
print("=" * 70)
for game_key in test_games:
    obs = eval_env.reset(game=game_key, strategy="tit_for_tat")
    prompt_text = prompt_builder.build(obs)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt_text},
    ]
    formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    output = pipe(formatted)[0]["generated_text"][len(formatted):].strip()
    move = parse_action(output, obs.available_actions)

    # Play full episode with this move
    obs = eval_env.reset(game=game_key, strategy="tit_for_tat")
    while not obs.done:
        obs = eval_env.step(GameAction(action=move))
    coop = _compute_cooperation_rate(obs)
    r = episode_reward(obs.player_score, obs.opponent_score, coop, obs.current_round)

    game_name = GAMES[game_key].name
    print(f"{game_name:<30s} {move:<15s} {r:>15.3f}")